**DATA COLLECTION**

In [ ]:


import argparse
import csv
import logging
import os
import re
import sys
import statistics
import time
import unicodedata
from datetime import datetime, timezone
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; EVResearchScraper/2.0; +contact@example.com)",
    "Accept-Language": "en-US,en;q=0.9",
}
REQUEST_DELAY = 2.0   # seconds between page requests
TIMEOUT = 20
RETRIES = 3

# ETB per 1 USD. Placeholder: UPDATE to the current rate before analysis.
USD_TO_ETB = 155.0

# Canonical model -> regex aliases (applied to normalized, lowercased text)
MODELS = {
    "BYD Seagull": [r"\bbyd\s*seagull\b", r"\bseagull\b"],
    "BYD Yuan Up": [r"\bbyd\s*yuan\s*up\b", r"\byuan\s*up\b"],
    "Changan Benben E-Star": [r"\bbenben\s*e[\s-]?star\b", r"\bbenben\b"],
    "Toyota bZ4X": [r"\bbz\s*4\s*x\b"],
    "Volkswagen ID.4": [r"\b(vw\s*)?id\.?\s*4\b"],
    "Volkswagen ID.6": [r"\b(vw\s*)?id\.?\s*6\b"],
}

# Phrases that indicate a distributor / general agent / dealer seller.
DISTRIBUTOR_KEYWORDS = [
    "general agent", "general market", "royal car", "authorized", "authorised",
    "official", "distributor", "showroom", "dealership", "dealer", "gezat",
]

# Official evidence: model pages/listings found on a distributor's own site.
# Gezat's homepage lists BYD Seagull and BYD Yuan Up; it also links to
# byd.com.et for the Seagull. Changan, VW and Toyota models are not named on
# the homepage, so they are left empty until verified.
OFFICIAL_EVIDENCE = {
    "BYD Seagull": ["https://ev.gezat.et/byd/seagull",
                    "https://www.byd.com.et/vehicles/byd/seagull/"],
    "BYD Yuan Up": ["https://ev.gezat.et/ (BYD lineup)"],
}

# Scoring weights (must sum to 1.0)
WEIGHTS = {
    "popularity": 0.30,
    "distributor": 0.25,
    "price": 0.25,
    "year": 0.10,
    "availability": 0.10,
}

LOCATION_RULES = [
    (r"addis|bole|megenagna|piassa", "Addis Ababa"),
    (r"adama|nazret", "Adama"),
    (r"bishoftu|debre ?zeit", "Bishoftu"),
    (r"hawassa|hawasa", "Hawassa"),
    (r"bahir ?dar", "Bahir Dar"),
    (r"mekel[ei]", "Mekelle"),
    (r"dire ?dawa", "Dire Dawa"),
    (r"jimma", "Jimma"),
    (r"gondar", "Gondar"),
]

# Each source: "urls" may contain {page}. Set "enabled": True only after the
# selectors are verified. For cards that are themselves links (Adey), set
# "card_is_link": True and only "item" is needed.
SOURCES = [
    {
        "name": "adey_market",
        "enabled": True,
        "card_is_link": True,
        "item": "a[href^='/item/']",
        "urls": [
            "https://adey-market.com/categories/cars/byd-yuan-up-atto-2?page={page}",
            "https://adey-market.com/categories/cars/byd-seagull-dolphin-mini-atto-1?page={page}",
            "https://adey-market.com/categories/cars/changan-benben-e-star?page={page}",
            "https://adey-market.com/categories/cars/toyota-bz-series-bz4x-bz3-bz3x-bz5?page={page}",
            "https://adey-market.com/categories/cars/volkswagen-id-4?page={page}",
            "https://adey-market.com/categories/cars/volkswagen-id-4-crozz-faw-vw?page={page}",
            "https://adey-market.com/categories/cars/volkswagen-id-6?page={page}",
        ],
    },
    {
        # Distributor site. Its inventory may be empty; its model pages are the
        # main value here, and those feed OFFICIAL_EVIDENCE above.
        "name": "gezat",
        "enabled": False,
        "is_official": True,
        "item": "",
        "title": "", "price": "", "location": "", "extra": "", "link": "a",
        "urls": ["https://gezat.et/cars-for-sale/electric"],
    },
    {
        # TODO: blocked for automated access during setup. Inspect the page in
        # a browser, set "item" and the field selectors, then set enabled True.
        "name": "mekina",
        "enabled": False,
        "item": "", "title": "", "price": "", "location": "", "extra": "", "link": "a",
        "urls": ["https://www.mekina.net/cars/search?page={page}"],
    },
    {
        # TODO: blocked for automated access during setup. Jiji may also render
        # listings with JavaScript, in which case a browser tool such as
        # Playwright is needed.
        "name": "jiji",
        "enabled": False,
        "item": "", "title": "", "price": "", "location": "", "extra": "", "link": "a",
        "urls": ["https://jiji.com.et/cars?page={page}"],
    },
]

RAW_COLS = [
    "scraped_at", "site", "model", "title", "price_raw", "price_value", "currency",
    "price_etb", "model_year", "availability", "is_distributor",
    "distributor_evidence", "location", "listing_url",
]
DEDUP_COLS = RAW_COLS + ["duplicate_count", "sites"]
SCORE_COLS = [
    "rank", "model", "score", "confidence", "listings", "priced_listings",
    "distributor_share", "official_evidence", "median_price_etb",
    "median_model_year", "available_share", "popularity_score",
    "distributor_score", "price_score", "year_score", "availability_score",
]

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("ev_scraper")

# ---------------------------------------------------------------------------
# Text and parsing helpers
# ---------------------------------------------------------------------------


def normalize(text: str) -> str:
    """NFKC folds fancy Unicode (e.g. Telegram's bold 𝗚𝗘𝗡𝗘𝗥𝗔𝗟) to plain ASCII."""
    return unicodedata.normalize("NFKC", text or "").lower()


def clean(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def match_model(text: str) -> str:
    for canonical, patterns in MODELS.items():
        if any(re.search(p, text) for p in patterns):
            return canonical
    return ""


NUM = r"(\d[\d,]*(?:\.\d+)?)\s*(million|thousand|m|k)?"
MULT = {"million": 1e6, "m": 1e6, "thousand": 1e3, "k": 1e3}
PRICE_RES = [
    # "3,200,000 etb", "2.85 million birr", "3.2m etb"
    (re.compile(NUM + r"\s*(?:etb|birr)\b"), "ETB", 1, 2),
    # "etb 2.85 million", "birr 450k"
    (re.compile(r"(?:etb|birr)\s*" + NUM), "ETB", 1, 2),
    # "$38,500", "usd 38500"
    (re.compile(r"(?:\$|usd)\s*(\d[\d,]*(?:\.\d+)?)"), "USD", 1, None),
    # "38,500 usd"
    (re.compile(NUM + r"\s*usd\b"), "USD", 1, 2),
]


def parse_price(text: str):
    """Return (value, currency, matched_text). Value is None if no price found."""
    t = normalize(text)
    for rx, currency, g_num, g_mult in PRICE_RES:
        m = rx.search(t)
        if m:
            value = float(m.group(g_num).replace(",", ""))
            if g_mult:
                value *= MULT.get(m.group(g_mult) or "", 1)
            return value, currency, m.group(0)
    return None, "", ""


def to_etb(value, currency):
    if value is None:
        return None
    return value * USD_TO_ETB if currency == "USD" else value


def parse_year(text: str) -> str:
    m = re.search(r"(?<!\d)(199\d|20[0-3]\d)(?!\d)", text)
    return m.group(1) if m else ""


def parse_availability(text: str) -> str:
    rules = [
        (r"\bsold\b", "Sold"),
        (r"\breserved\b", "Reserved"),
        (r"\bpre[\s-]?order\b", "Pre-order"),
        (r"\bout of stock\b|\bunavailable\b", "Out of stock"),
        (r"\bin stock\b|\bavailable\b|\bfor sale\b|\bon sale\b", "Available"),
    ]
    for pattern, label in rules:
        if re.search(pattern, text):
            return label
    return "Unknown"


def parse_location(text: str) -> str:
    t = normalize(text)
    for pattern, label in LOCATION_RULES:
        if re.search(pattern, t):
            return label
    return ""


def distributor_hit(text: str) -> str:
    t = normalize(text)
    for kw in DISTRIBUTOR_KEYWORDS:
        if kw in t:
            return kw
    return ""


# ---------------------------------------------------------------------------
# Scraping
# ---------------------------------------------------------------------------


def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def fetch(session: requests.Session, url: str):
    for attempt in range(1, RETRIES + 1):
        try:
            r = session.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            return r.text
        except requests.RequestException as exc:
            log.warning("Fetch failed (%d/%d) %s: %s", attempt, RETRIES, url, exc)
            time.sleep(REQUEST_DELAY * attempt)
    return None


_robots: dict = {}


def robots_allows(url: str) -> bool:
    p = urlparse(url)
    root = f"{p.scheme}://{p.netloc}"
    if root not in _robots:
        rp = robotparser.RobotFileParser()
        rp.set_url(root + "/robots.txt")
        try:
            rp.read()
        except Exception:
            rp.allow_all = True
        _robots[root] = rp
    return _robots[root].can_fetch(HEADERS["User-Agent"], url)


def extract_listing(el, src: dict, page_url: str):
    """Turn one card element into a raw listing dict, or None if not a target model."""
    if src.get("card_is_link"):
        full_text = clean(el.get_text(" ", strip=True))
        title = full_text[:200]
        href = el.get("href", "")
        price_text = full_text
        location_text = full_text
        extra_text = ""
    else:
        def sel_text(sel):
            node = el.select_one(sel) if sel else None
            return clean(node.get_text(" ", strip=True)) if node else ""

        full_text = clean(el.get_text(" ", strip=True))
        title = sel_text(src.get("title")) or full_text[:200]
        price_text = sel_text(src.get("price")) or full_text
        location_text = sel_text(src.get("location")) or full_text
        extra_text = sel_text(src.get("extra"))
        link = el.select_one(src["link"]) if src.get("link") else el.find("a", href=True)
        href = link.get("href", "") if link else ""

    searchable = normalize(f"{full_text} {extra_text}")
    model = match_model(searchable)
    if not model:
        return None

    value, currency, price_raw = parse_price(price_text)
    etb = to_etb(value, currency)

    dist = distributor_hit(searchable)
    if src.get("is_official"):
        dist = dist or f"official site: {src['name']}"

    return {
        "scraped_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "site": src["name"],
        "model": model,
        "title": title,
        "price_raw": price_raw,
        "price_value": "" if value is None else f"{value:.0f}",
        "currency": currency,
        "price_etb": "" if etb is None else f"{etb:.0f}",
        "model_year": parse_year(title) or parse_year(full_text),
        "availability": parse_availability(searchable),
        "is_distributor": "yes" if dist else "no",
        "distributor_evidence": dist,
        "location": parse_location(location_text),
        "listing_url": urljoin(page_url, href) if href else "",
    }


def scrape_source(session, src: dict, max_pages: int, seen_urls: set, ignore_robots: bool = False) -> list:
    if not src.get("enabled"):
        log.warning("[%s] disabled (selectors not configured), skipping", src["name"])
        return []

    out = []
    for template in src["urls"]:
        paged = "{page}" in template
        for page in range(1, (max_pages if paged else 1) + 1):
            url = template.format(page=page) if paged else template
            if not ignore_robots and not robots_allows(url):
                log.warning("robots.txt disallows %s, skipping", url)
                break
            html = fetch(session, url)
            if html is None:
                break
            soup = BeautifulSoup(html, "html.parser")
            cards = soup.select(src["item"])
            log.info("[%s] %s -> %d cards", src["name"], url, len(cards))
            if not cards:
                break
            for el in cards:
                row = extract_listing(el, src, url)
                if row and row["listing_url"] not in seen_urls:
                    seen_urls.add(row["listing_url"])
                    out.append(row)
            if paged:
                time.sleep(REQUEST_DELAY)
    return out


# ---------------------------------------------------------------------------
# Deduplication
# ---------------------------------------------------------------------------


def dup_key(r: dict) -> str:
    """
    Same model, year, price (rounded to 5,000 ETB) and location = same car.
    Without a price, fall back to a normalized title.
    """
    if r["price_etb"]:
        price_part = str(round(float(r["price_etb"]) / 5000) * 5000)
    else:
        price_part = re.sub(r"[^a-z0-9]", "", normalize(r["title"]))[:60]
    return "|".join([r["model"], r["model_year"], price_part, normalize(r["location"])])


def dedupe(rows: list) -> list:
    merged: dict = {}
    order: list = []
    for r in rows:
        key = dup_key(r)
        if key not in merged:
            kept = dict(r, duplicate_count="1", sites=r["site"])
            merged[key] = kept
            order.append(key)
            continue
        kept = merged[key]
        kept["duplicate_count"] = str(int(kept["duplicate_count"]) + 1)
        if r["site"] not in kept["sites"].split("; "):
            kept["sites"] += f"; {r['site']}"
        if r["is_distributor"] == "yes" and kept["is_distributor"] == "no":
            kept["is_distributor"] = "yes"
            kept["distributor_evidence"] = r["distributor_evidence"]
    return [merged[k] for k in order]


# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------


def score_models(rows: list) -> list:
    groups = {m: [r for r in rows if r["model"] == m] for m in MODELS}
    max_n = max([len(v) for v in groups.values()] + [1])

    med_price, med_year = {}, {}
    for m, v in groups.items():
        prices = [float(r["price_etb"]) for r in v if r["price_etb"]]
        years = [int(r["model_year"]) for r in v if r["model_year"]]
        if len(prices) >= 3:
            med_price[m] = statistics.median(prices)
        if years:
            med_year[m] = statistics.median(years)

    best_price = min(med_price.values()) if med_price else None
    yr_lo = min(med_year.values()) if med_year else None
    yr_hi = max(med_year.values()) if med_year else None

    results = []
    for m, v in groups.items():
        n = len(v)
        if n == 0:
            results.append({"model": m, "score": 0.0, "confidence": "no data", "listings": 0})
            continue

        priced = sum(1 for r in v if r["price_etb"])
        dist_share = sum(r["is_distributor"] == "yes" for r in v) / n
        official = 1.0 if OFFICIAL_EVIDENCE.get(m) else 0.0

        parts = {
            "popularity": n / max_n,
            "distributor": 0.7 * dist_share + 0.3 * official,
            # Cheapest model gets 1.0; others scored by how far above it they sit.
            "price": min(1.0, best_price / med_price[m]) if m in med_price else 0.5,
            "year": ((med_year[m] - yr_lo) / (yr_hi - yr_lo)
                     if m in med_year and yr_hi > yr_lo else (1.0 if m in med_year else 0.5)),
            "availability": 1 - sum(r["availability"] in ("Sold", "Reserved", "Out of stock")
                                    for r in v) / n,
        }
        score = 100 * sum(WEIGHTS[k] * parts[k] for k in WEIGHTS)

        if n >= 20 and priced >= 5:
            confidence = "high"
        elif n >= 5 and priced >= 3:
            confidence = "medium"
        else:
            confidence = "low"

        results.append({
            "model": m,
            "score": round(score, 1),
            "confidence": confidence,
            "listings": n,
            "priced_listings": priced,
            "distributor_share": round(dist_share, 2),
            "official_evidence": "yes" if official else "no",
            "median_price_etb": f"{med_price[m]:.0f}" if m in med_price else "",
            "median_model_year": f"{med_year[m]:.0f}" if m in med_year else "",
            "available_share": round(parts["availability"], 2),
            "popularity_score": round(parts["popularity"], 2),
            "distributor_score": round(parts["distributor"], 2),
            "price_score": round(parts["price"], 2),
            "year_score": round(parts["year"], 2),
            "availability_score": round(parts["availability"], 2),
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    for i, r in enumerate(results, 1):
        r["rank"] = i
    return results


# ---------------------------------------------------------------------------
# CSV I/O
# ---------------------------------------------------------------------------


def write_csv(rows: list, columns: list, path: str) -> None:
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    log.info("Wrote %d rows to %s", len(rows), path)


def read_csv(path: str) -> list:
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


# ---------------------------------------------------------------------------
# Self-test (offline)
# ---------------------------------------------------------------------------

SAMPLE_HTML = """
<a href="/item/1001-byd-seagull-2024">BYD Seagull 2024 3,200,000 ETB Royal Car's Ethiopia · Addis Ababa · 2h ago</a>
<a href="/item/1002-byd-seagull-2024">BYD Seagull 2024 3,200,000 ETB Some Seller · Addis Ababa · 3h ago</a>
<a href="/item/1003-id6">VW ID.6 2023 Price on request Some Seller · Addis Ababa</a>
<a href="/item/1004-bz4x">Toyota bZ4X 2023 $42,000 · Adama</a>
<a href="/item/1005-bz3">Toyota bZ3 2024 2,500,000 ETB · Addis Ababa</a>
<a href="/item/1006-yuan-up">BYD Yuan Up 2025 3,800,000 ETB 𝗚𝗘𝗡𝗘𝗥𝗔𝗟 𝗠𝗔𝗥𝗞𝗘𝗧 · Addis Ababa</a>
"""


def selftest() -> None:
    src = SOURCES[0]
    soup = BeautifulSoup(SAMPLE_HTML, "html.parser")
    raw = [extract_listing(a, src, "https://adey-market.com/categories/cars")
           for a in soup.select(src["item"])]
    raw = [r for r in raw if r]
    assert len(raw) == 5, f"expected 5 target rows before dedupe, got {len(raw)}"

    unique = dedupe(raw)
    assert len(unique) == 4, f"expected 4 unique listings, got {len(unique)}"

    by_model = {r["model"]: r for r in unique}
    seagull = by_model["BYD Seagull"]
    assert seagull["duplicate_count"] == "2" and seagull["price_etb"] == "3200000"
    assert seagull["model_year"] == "2024" and seagull["is_distributor"] == "yes"

    assert by_model["Volkswagen ID.6"]["price_etb"] == ""       # price on request
    assert by_model["Toyota bZ4X"]["price_etb"] == f"{42000 * USD_TO_ETB:.0f}"
    assert by_model["BYD Yuan Up"]["is_distributor"] == "yes"   # unicode-bold banner
    assert by_model["BYD Yuan Up"]["availability"] == "Unknown"

    scores = score_models(unique)
    assert len(scores) == 6
    assert [s["rank"] for s in scores] == list(range(1, 7))

    print("Self-test passed.\n")
    print(f"{'rank':<5}{'model':<24}{'score':<8}{'listings':<10}{'confidence'}")
    for s in scores:
        print(f"{s['rank']:<5}{s['model']:<24}{s['score']:<8}"
              f"{s['listings']:<10}{s['confidence']}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------


def main() -> None:
    ap = argparse.ArgumentParser(description="Scrape, dedupe and rank Ethiopian EV listings.")
    ap.add_argument("--out-dir", default="ev_output")
    ap.add_argument("--max-pages", type=int, default=3, help="pages per paged URL")
    ap.add_argument("--from-raw", help="skip scraping; process an existing raw CSV")
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--ignore-robots", action="store_true", default=True, help="ignore robots.txt constraints")

    # Safeguard for running inside Jupyter notebooks (which pass unexpected internal flags)
    if any(arg.startswith('-f') for arg in sys.argv):
        args = ap.parse_args([])
    else:
        args = ap.parse_args()

    if args.selftest:
        selftest()
        return

    os.makedirs(args.out_dir, exist_ok=True)

    if args.from_raw:
        raw = read_csv(args.from_raw)
    else:
        session = make_session()
        seen: set = set()
        raw = []
        for src in SOURCES:
            try:
                raw.extend(scrape_source(session, src, args.max_pages, seen, ignore_robots=args.ignore_robots))
            except Exception:
                log.exception("Failed scraping %s", src["name"])
        write_csv(raw, RAW_COLS, os.path.join(args.out_dir, "raw_listings.csv"))

    unique = dedupe(raw)
    write_csv(unique, DEDUP_COLS, os.path.join(args.out_dir, "listings_deduped.csv"))

    scores = score_models(unique)
    write_csv(scores, SCORE_COLS, os.path.join(args.out_dir, "model_scores.csv"))
    top3 = [s for s in scores if s["score"]][:3]
    write_csv(top3, SCORE_COLS, os.path.join(args.out_dir, "top3_models.csv"))

    print("\nTop 3 models:")
    for s in top3:
        print(f"  {s['rank']}. {s['model']}  score={s['score']}  "
              f"listings={s['listings']}  confidence={s['confidence']}")


if __name__ == "__main__":
    main()


Top 3 models:
  1. BYD Seagull  score=62.2  listings=39  confidence=high
  2. BYD Yuan Up  score=61.3  listings=38  confidence=high
  3. Volkswagen ID.6  score=53.1  listings=67  confidence=high


In [3]:
import re
import pandas as pd

raw = pd.read_csv("ev_output/raw_listings.csv", dtype=str)

# 1. Drop rows without a target model or a listing URL.
df = raw.dropna(subset=["model"]).copy()
df = df[df["model"].str.strip() != ""]

# 2. Prices: keep numbers only, drop impossible values.
df["price_etb"] = pd.to_numeric(df["price_etb"], errors="coerce")
df = df[df["price_etb"].between(100_000, 100_000_000) | df["price_etb"].isna()]

# 3. Model year: four digits in range, otherwise blank.
df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce")
df.loc[~df["model_year"].between(1990, 2039), "model_year"] = pd.NA
df["model_year"] = df["model_year"].astype("Int64")

# 4. Availability: map variants to fixed labels.
label_map = {
    "in stock": "Available", "available": "Available",
    "sold": "Sold", "reserved": "Reserved",
    "pre-order": "Pre-order", "preorder": "Pre-order",
    "out of stock": "Out of stock", "unavailable": "Out of stock",
}
df["availability"] = (
    df["availability"].fillna("Unknown").str.strip().str.lower()
      .map(label_map).fillna("Unknown")
)

# 5. Text fields: trim spaces and collapse repeats.
for col in ["title", "location", "site"]:
    df[col] = df[col].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

# 6. Location: one spelling per city.
df["location"] = df["location"].replace({"Addis": "Addis Ababa", "addis ababa": "Addis Ababa"})

# 7. Listing URL: drop rows without one, then remove exact URL duplicates.
df = df[df["listing_url"].fillna("") != ""]
df = df.drop_duplicates(subset=["listing_url"])

df.to_csv("ev_output/raw_clean.csv", index=False, encoding="utf-8-sig")
print(f"{len(raw)} rows in, {len(df)} rows out")

317 rows in, 317 rows out


**CHECKING RAW DATA**

In [4]:
import re
import pandas as pd

raw = pd.read_csv("ev_output/raw_listings.csv", dtype=str)

# 1. Drop rows without a target model or a listing URL.
df = raw.dropna(subset=["model"]).copy()
df = df[df["model"].str.strip() != ""]

# 2. Prices: keep numbers only, drop impossible values.
df["price_etb"] = pd.to_numeric(df["price_etb"], errors="coerce")
df = df[df["price_etb"].between(100_000, 100_000_000) | df["price_etb"].isna()]

# 3. Model year: four digits in range, otherwise blank.
df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce")
df.loc[~df["model_year"].between(1990, 2039), "model_year"] = pd.NA
df["model_year"] = df["model_year"].astype("Int64")

# 4. Availability: map variants to fixed labels.
label_map = {
    "in stock": "Available", "available": "Available",
    "sold": "Sold", "reserved": "Reserved",
    "pre-order": "Pre-order", "preorder": "Pre-order",
    "out of stock": "Out of stock", "unavailable": "Out of stock",
}
df["availability"] = (
    df["availability"].fillna("Unknown").str.strip().str.lower()
      .map(label_map).fillna("Unknown")
)

# 5. Text fields: trim spaces and collapse repeats.
for col in ["title", "location", "site"]:
    df[col] = df[col].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

# 6. Location: one spelling per city.
df["location"] = df["location"].replace({"Addis": "Addis Ababa", "addis ababa": "Addis Ababa"})

# 7. Listing URL: drop rows without one, then remove exact URL duplicates.
df = df[df["listing_url"].fillna("") != ""]
df = df.drop_duplicates(subset=["listing_url"])

df.to_csv("ev_output/raw_clean.csv", index=False, encoding="utf-8-sig")
print(f"{len(raw)} rows in, {len(df)} rows out")

317 rows in, 317 rows out


In [14]:
print(df["price_etb"].describe())

count    1.650000e+02
mean     5.536206e+06
std      1.656515e+06
min      2.540000e+05
25%      4.500000e+06
50%      5.800000e+06
75%      6.400000e+06
max      9.500000e+06
Name: price_etb, dtype: float64


**REMOVING DUPLICATES**

In [6]:
df["price_bucket"] = (df["price_etb"] / 5000).round() * 5000

dupes = df[df.duplicated(
    subset=["model", "model_year", "price_bucket", "location"], keep=False
)]
print(f"{len(dupes)} rows are duplicates of another row")

df = df.drop_duplicates(subset=["model", "model_year", "price_bucket", "location"])

188 rows are duplicates of another row


**CLEANED DATA**

In [7]:
print(df["model"].value_counts())
print(df["price_etb"].describe())
print(df["model_year"].value_counts(dropna=False).sort_index())
print(df["availability"].value_counts())

model
Volkswagen ID.6          51
Volkswagen ID.4          43
BYD Yuan Up              32
BYD Seagull              30
Toyota bZ4X              26
Changan Benben E-Star     5
Name: count, dtype: int64
count    1.650000e+02
mean     5.536206e+06
std      1.656515e+06
min      2.540000e+05
25%      4.500000e+06
50%      5.800000e+06
75%      6.400000e+06
max      9.500000e+06
Name: price_etb, dtype: float64
model_year
2022    40
2023    18
2024    22
2025    16
2026    20
<NA>    71
Name: count, dtype: Int64
availability
Unknown      182
Available      5
Name: count, dtype: int64


**PRIORITY SCORING BASED ON CLEAN DATA**

In [13]:
import argparse
import json
import sys

import pandas as pd

# ---------------------------------------------------------------------------
# Configuration: edit these to change the model
# ---------------------------------------------------------------------------

TARGET_MODELS = [
    "BYD Seagull",
    "BYD Yuan Up",
    "Changan Benben E-Star",
    "Toyota bZ4X",
    "Volkswagen ID.4",
    "Volkswagen ID.6",
 ]

WEIGHTS = {
    "popularity": 0.30,
    "distributor": 0.25,
    "price": 0.25,
    "model_year": 0.10,
    "availability": 0.10,
}

# Within the distributor signal: share of listings from distributors counts
# this much, and official evidence counts the rest.
DISTRIBUTOR_LISTING_SHARE = 0.7
OFFICIAL_EVIDENCE_SHARE = 0.3

# Models with an official listing on a distributor's own site (see the scraper's
# OFFICIAL_EVIDENCE). Add or remove names as you verify them.
OFFICIAL_EVIDENCE = {"BYD Seagull", "BYD Yuan Up"}

# A model needs at least this many priced listings to be ranked on price.
MIN_PRICED_LISTINGS = 3

# Confidence levels, based on how much data each model has.
HIGH_CONFIDENCE = {"min_listings": 20, "min_priced": 5}
MEDIUM_CONFIDENCE = {"min_listings": 5, "min_priced": 3}

UNAVAILABLE_LABELS = {"Sold", "Reserved", "Out of stock"}

REQUIRED_COLUMNS = ["model", "price_etb", "model_year", "availability", "is_distributor"]


# ---------------------------------------------------------------------------
# Data checks
# ---------------------------------------------------------------------------


def load_listings(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        sys.exit(f"Input is missing required columns: {missing}")

    df["price_etb"] = pd.to_numeric(df["price_etb"], errors="coerce")
    df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce")
    df["availability"] = df["availability"].fillna("Unknown")
    df["is_distributor"] = df["is_distributor"].fillna("no").str.lower()
    return df


def validate_weights(weights: dict) -> None:
    total = sum(weights.values())
    if abs(total - 1.0) > 1e-6:
        sys.exit(f"Weights must sum to 1.0 (currently {total:.3f})")
    unknown = set(weights) - set(WEIGHTS)
    if unknown:
        sys.exit(f"Unknown weight names: {sorted(unknown)}")


# ---------------------------------------------------------------------------
# Signals
# ---------------------------------------------------------------------------


def compute_signals(df: pd.DataFrame) -> pd.DataFrame:
    """One row per target model with raw stats and 0-1 signal scores."""
    rows = []
    for model in TARGET_MODELS:
        m = df[df["model"] == model]
        n = len(m)
        priced = m["price_etb"].dropna()
        years = m["model_year"].dropna()
        dist_share = (m["is_distributor"] == "yes").mean() if n else 0.0
        unavailable = m["availability"].isin(UNAVAILABLE_LABELS).mean() if n else 0.0

        rows.append({
            "model": model,
            "listings": n,
            "priced_listings": len(priced),
            "median_price_etb": priced.median() if len(priced) >= MIN_PRICED_LISTINGS else None,
            "median_model_year": years.median() if len(years) >= MIN_PRICED_LISTINGS else None,
            "distributor_share": dist_share,
            "official_evidence": model in OFFICIAL_EVIDENCE,
            "available_share": 1 - unavailable if n else 0.0,
        })
    stats = pd.DataFrame(rows)

    max_n = stats["listings"].max() or 1
    stats["popularity"] = stats["listings"] / max_n

    stats["distributor"] = (
        DISTRIBUTOR_LISTING_SHARE * stats["distributor_share"]
        + OFFICIAL_EVIDENCE_SHARE * stats["official_evidence"].astype(float)
    )

    best_price = stats["median_price_etb"].min()
    stats["price"] = [
        min(1.0, best_price / p) if pd.notna(p) and pd.notna(best_price) else 0.5
        for p in stats["median_price_etb"]
    ]

    yr = stats["median_model_year"].dropna()
    lo, hi = (yr.min(), yr.max()) if len(yr) else (None, None)
    stats["model_year"] = [
        1.0 if lo is None and pd.notna(y) and hi is None else
        ((y - lo) / (hi - lo) if pd.notna(y) and hi and hi > lo else
         (1.0 if pd.notna(y) else 0.5))
        for y in stats["median_model_year"]
    ]

    stats["availability"] = stats["available_share"]
    return stats


def confidence(listings: int, priced: int) -> str:
    if listings >= HIGH_CONFIDENCE["min_listings" ] and priced >= HIGH_CONFIDENCE["min_priced"]:
        return "high"
    if listings >= MEDIUM_CONFIDENCE["min_listings"] and priced >= MEDIUM_CONFIDENCE["min_priced"]:
        return "medium"
    return "low"


def explain(row: pd.Series) -> str:
    """Plain-language reasons for a model's score."""
    notes = []
    notes.append(f"{row['listings']} listings")
    if row["popularity"] >= 0.8:
        notes.append("among the most listed")
    elif row["popularity"] < 0.2:
        notes.append("few listings")

    if row["official_evidence"]:
        notes.append("official distributor evidence")
    if row["distributor_share"] >= 0.5:
        notes.append(f"{row['distributor_share']:.0%} from distributors")
    elif row["distributor_share"] == 0:
        notes.append("no distributor listings found")

    if pd.isna(row["median_price_etb"]):
        notes.append("not enough priced listings to rank on price")
    else:
        notes.append(f"median price {row['median_price_etb']:,.0f} ETB")

    if pd.notna(row["median_model_year"]):
        notes.append(f"median model year {row['median_model_year']:.0f}")

    if row["available_share"] < 0.5 and row["listings"] > 0:
        notes.append("many listings sold or reserved")
    return "; ".join(notes)


# ---------------------------------------------------------------------------
# Ranking
# ---------------------------------------------------------------------------


def rank_models(df: pd.DataFrame, weights: dict) -> pd.DataFrame:
    stats = compute_signals(df)

    stats["score"] = 100 * sum(stats[k] * w for k, w in weights.items())
    stats["confidence"] = [confidence(n, p) for n, p in
                           zip(stats["listings"], stats["priced_listings"])]
    stats["explanation"] = stats.apply(explain, axis=1)

    stats = stats.sort_values("score", ascending=False).reset_index(drop=True)
    stats.insert(0, "rank", stats.index + 1)
    return stats


def print_report(ranked: pd.DataFrame, weights: dict, top: int = 3) -> None:
    print("\nWeights used:", ", ".join(f"{k}={v:.2f}" for k, v in weights.items()))
    print("\nTop priority models")
    print("-" * 70)
    for _, r in ranked.head(top).iterrows():
        print(f"{int(r['rank'])}. {r['model']}  score {r['score']:.1f}  "
              f"[confidence: {r['confidence']}]")
        print(f"   {r['explanation']}")
    print("\nAll models")
    print("-" * 70)
    cols = ["rank", "model", "score", "confidence", "listings"]
    print(ranked[cols].to_string(index=False, float_format=lambda x: f"{x:.1f}"))


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------


def main() -> None:
    ap = argparse.ArgumentParser(description="Rank EV models by priority.")
    ap.add_argument("input", nargs="?", help="cleaned listings CSV")
    ap.add_argument("--weights", help="JSON file overriding the default weights")
    ap.add_argument("--out", default="priority_ranking.csv", help="output CSV")
    ap.add_argument("--demo", action="store_true", help="use built-in sample data")

    # Jupyter safety: If run from Colab directly without CLI args, default to raw_clean.csv
    if any(arg.startswith('-f') for arg in sys.argv):
        args = ap.parse_args(["ev_output/raw_clean.csv"])
    else:
        args = ap.parse_args()

    weights = dict(WEIGHTS)
    if args.weights:
        with open(args.weights) as f:
            weights.update(json.load(f))
    validate_weights(weights)

    if args.demo:
        # Fallback inline sample
        rows = [
            ("BYD Yuan Up", 3400000, 2024, "Available", "yes"),
            ("BYD Yuan Up", 3600000, 2025, "Unknown", "yes"),
            ("BYD Yuan Up", 3500000, 2024, "Available", "no"),
            ("BYD Seagull", 2850000, 2024, "Available", "yes"),
            ("Toyota bZ4X", 6500000, 2023, "Unknown", "no"),
            ("Volkswagen ID.4", 5900000, 2022, "Unknown", "no")
        ]
        df = pd.DataFrame(rows, columns=REQUIRED_COLUMNS)
    elif args.input:
        df = load_listings(args.input)
    else:
        ap.error("provide an input CSV or use --demo")

    ranked = rank_models(df, weights)
    ranked.to_csv(args.out, index=False, encoding="utf-8-sig")
    print_report(ranked, weights)
    print(f"\nSaved to {args.out}")


if __name__ == "__main__":
    main()


Weights used: popularity=0.30, distributor=0.25, price=0.25, model_year=0.10, availability=0.10

Top priority models
----------------------------------------------------------------------
1. BYD Seagull  score 62.0  [confidence: high]
   49 listings; official distributor evidence; median price 4,100,000 ETB; median model year 2025
2. BYD Yuan Up  score 60.0  [confidence: high]
   52 listings; official distributor evidence; median price 5,500,000 ETB; median model year 2025
3. Volkswagen ID.6  score 54.1  [confidence: high]
   101 listings; among the most listed; median price 6,750,000 ETB; median model year 2022

All models
----------------------------------------------------------------------
 rank                 model  score confidence  listings
    1           BYD Seagull   62.0       high        49
    2           BYD Yuan Up   60.0       high        52
    3       Volkswagen ID.6   54.1       high       101
    4 Changan Benben E-Star   48.1     medium        10
    5       Volk